# HW03 深度学习作业 3

- 学号：20234080305
- 姓名：whf
- 文件：`HW03-20234080305-whf.ipynb`

本 notebook 按 `HW03.pdf` 的题目顺序完成：卷积与池化层、LeNet/AlexNet/VGG/NiN、Inception/批量归一化/残差网络、图像增广/微调/样式迁移、目标检测与训练技巧。理论题给出计算过程，编程题给出可直接运行的实现和中文输出。

In [1]:
# 导入本作业需要的基础库，并设置随机种子，保证示例输出尽量可复现。
import math
import numpy as np
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(42)

print("基础库导入完成。")
print(f"NumPy 版本：{np.__version__}")
print(f"PyTorch 版本：{torch.__version__}")
print("torchvision 图像增广模块已导入。")

基础库导入完成。
NumPy 版本：2.4.4
PyTorch 版本：2.12.0+cpu
torchvision 图像增广模块已导入。


## 2 卷积和池化层

### 2.1 理论计算题

输入图像尺寸为 $3 \times 32 \times 32$，卷积核数量为 16，每个卷积核大小为 $3 \times 5 \times 5$，填充 $p=2$，步幅 $s=2$。

卷积输出的空间尺寸计算公式为

$$
H_{out}=W_{out}=\left\lfloor \frac{H+2p-k}{s}\right\rfloor + 1.
$$

代入 $H=W=32, p=2, k=5, s=2$：

$$
H_{out}=W_{out}=\left\lfloor \frac{32+2\times2-5}{2}\right\rfloor+1
=\left\lfloor \frac{31}{2}\right\rfloor+1=16.
$$

因此输出特征图尺寸为

$$
16 \times 16 \times 16.
$$

其中第一个 16 是输出通道数，后两个 16 是高和宽。

单个输出通道的一个像素值，需要把一个 $3 \times 5 \times 5$ 的卷积核与对应输入窗口逐元素相乘后求和，因此乘法次数为

$$
3 \times 5 \times 5 = 75.
$$

所以答案为：输出尺寸是 $16 \times 16 \times 16$；单个输出像素需要 75 次乘法操作。

### 2.2 编程题：手动实现二维最大池化

下面只使用 NumPy 实现最大池化前向传播，不调用深度学习框架中的池化 API。函数支持：

- 输入为二维矩阵 `(H, W)`；
- 输入为三维张量 `(C, H, W)`；
- 输入为四维张量 `(N, C, H, W)`；
- `kernel_size`、`stride`、`padding` 可以是整数，也可以是二元组。

In [2]:
def _to_pair(value, name):
    """把整数或长度为 2 的序列统一转换成二元组。"""
    if isinstance(value, int):
        return value, value
    if isinstance(value, (tuple, list)) and len(value) == 2:
        return int(value[0]), int(value[1])
    raise ValueError(f"{name} 必须是整数或长度为 2 的元组。")


def max_pool2d_numpy(x, kernel_size, stride=None, padding=0):
    """
    使用 NumPy 手动实现二维最大池化前向传播。

    参数：
    x：输入数组，形状可以是 (H, W)、(C, H, W) 或 (N, C, H, W)。
    kernel_size：池化窗口大小。
    stride：池化步幅；如果为 None，则默认等于 kernel_size。
    padding：在高和宽两个方向补零的宽度。
    """
    x = np.asarray(x, dtype=float)
    kernel_h, kernel_w = _to_pair(kernel_size, "kernel_size")
    stride_h, stride_w = _to_pair(kernel_size if stride is None else stride, "stride")
    pad_h, pad_w = _to_pair(padding, "padding")

    if kernel_h <= 0 or kernel_w <= 0:
        raise ValueError("池化窗口大小必须为正数。")
    if stride_h <= 0 or stride_w <= 0:
        raise ValueError("步幅必须为正数。")
    if pad_h < 0 or pad_w < 0:
        raise ValueError("填充不能为负数。")

    # 统一转换为四维形状，便于同时处理批量维和通道维。
    original_ndim = x.ndim
    if original_ndim == 2:
        x_work = x[None, None, :, :]
    elif original_ndim == 3:
        x_work = x[None, :, :, :]
    elif original_ndim == 4:
        x_work = x
    else:
        raise ValueError("输入 x 的维度必须是 2、3 或 4。")

    # 最大池化的填充值设为负无穷，避免填充值影响最大值结果。
    x_padded = np.pad(
        x_work,
        pad_width=((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
        mode="constant",
        constant_values=-np.inf,
    )

    batch_size, channels, padded_h, padded_w = x_padded.shape
    out_h = (padded_h - kernel_h) // stride_h + 1
    out_w = (padded_w - kernel_w) // stride_w + 1
    if out_h <= 0 or out_w <= 0:
        raise ValueError("池化窗口过大，无法产生有效输出。")

    output = np.empty((batch_size, channels, out_h, out_w), dtype=float)

    # 逐窗口取最大值，完整模拟最大池化前向传播。
    for i in range(out_h):
        row_start = i * stride_h
        row_end = row_start + kernel_h
        for j in range(out_w):
            col_start = j * stride_w
            col_end = col_start + kernel_w
            window = x_padded[:, :, row_start:row_end, col_start:col_end]
            output[:, :, i, j] = window.max(axis=(2, 3))

    # 按输入维度还原输出形状。
    if original_ndim == 2:
        return output[0, 0]
    if original_ndim == 3:
        return output[0]
    return output

In [3]:
# 使用一个小矩阵验证手写最大池化函数。
example_matrix = np.array([
    [1, 3, 2, 0],
    [4, 6, 5, 1],
    [7, 2, 9, 8],
    [3, 1, 4, 2],
], dtype=float)

pooled_without_padding = max_pool2d_numpy(example_matrix, kernel_size=2, stride=2, padding=0)
pooled_with_padding = max_pool2d_numpy(example_matrix, kernel_size=3, stride=2, padding=1)

print("原始输入矩阵：")
print(example_matrix)
print("\n窗口为 2、步幅为 2、无填充时的最大池化结果：")
print(pooled_without_padding)
print("\n窗口为 3、步幅为 2、填充为 1 时的最大池化结果：")
print(pooled_with_padding)

原始输入矩阵：
[[1. 3. 2. 0.]
 [4. 6. 5. 1.]
 [7. 2. 9. 8.]
 [3. 1. 4. 2.]]

窗口为 2、步幅为 2、无填充时的最大池化结果：
[[6. 5.]
 [7. 9.]]

窗口为 3、步幅为 2、填充为 1 时的最大池化结果：
[[6. 6.]
 [7. 9.]]


## 3 LeNet、AlexNet、VGG 和 NiN

### 3.1 理论计算题

假设输入和输出通道数均为 $C$，且卷积层不带偏置。

1. 一个 $5 \times 5$ 卷积层的参数量为

$$
C_{out} \times C_{in} \times 5 \times 5 = C \times C \times 25 = 25C^2.
$$

2. 两个串联的 $3 \times 3$ 卷积层，每层参数量都是

$$
C \times C \times 3 \times 3 = 9C^2.
$$

因此两个串联 $3 \times 3$ 卷积层的总参数量为

$$
2 \times 9C^2 = 18C^2.
$$

结论：两个 $3 \times 3$ 卷积层可以获得等效的 $5 \times 5$ 感受野，同时参数量从 $25C^2$ 降为 $18C^2$，减少了 $7C^2$ 个参数，并且中间多了一次非线性激活，表达能力更强。

### 3.2 编程题：定义 NiN 块

NiN 块由一个普通卷积层和两个 $1 \times 1$ 卷积层组成，每个卷积层后面紧跟 ReLU 激活函数。

In [4]:
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """使用 torch.nn.Sequential 定义标准 NiN 块。"""
    return nn.Sequential(
        # 第一层使用题目指定的普通卷积，负责提取局部空间特征。
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        # 后两层使用 1 x 1 卷积，相当于在每个像素位置上做通道混合。
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


# 构造一个示例 NiN 块，并验证输入输出形状。
nin = nin_block(in_channels=3, out_channels=8, kernel_size=5, stride=1, padding=2)
dummy_image_batch = torch.randn(2, 3, 32, 32)
nin_output = nin(dummy_image_batch)

print("NiN 块层序：普通卷积 -> ReLU -> 1x1 卷积 -> ReLU -> 1x1 卷积 -> ReLU")
print(f"示例输入形状：{tuple(dummy_image_batch.shape)}")
print(f"示例输出形状：{tuple(nin_output.shape)}")
print("输出通道数已变为 out_channels=8，空间尺寸因 padding=2、stride=1 保持为 32 x 32。")

NiN 块层序：普通卷积 -> ReLU -> 1x1 卷积 -> ReLU -> 1x1 卷积 -> ReLU
示例输入形状：(2, 3, 32, 32)
示例输出形状：(2, 8, 32, 32)
输出通道数已变为 out_channels=8，空间尺寸因 padding=2、stride=1 保持为 32 x 32。


## 4 Inception、批量归一化和残差网络

### 4.1 理论计算题

给定小批量中的 4 个特征值：

$$
x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8.
$$

批量均值为

$$
\mu = \frac{2+4+6+8}{4}=5.
$$

批量方差为

$$
\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=5.
$$

由于 $\epsilon=0$，标准差为 $\sqrt{5}$。批量归一化输出为

$$
y_i = \gamma \frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}} + \beta
=2\frac{x_i-5}{\sqrt{5}}+1.
$$

因此：

$$
y_1 = 1-\frac{6}{\sqrt{5}} \approx -1.6833,
$$

$$
y_2 = 1-\frac{2}{\sqrt{5}} \approx 0.1056,
$$

$$
y_3 = 1+\frac{2}{\sqrt{5}} \approx 1.8944,
$$

$$
y_4 = 1+\frac{6}{\sqrt{5}} \approx 3.6833.
$$

In [5]:
# 用代码复核批量归一化理论题的计算结果。
x = np.array([2, 4, 6, 8], dtype=float)
gamma = 2.0
beta = 1.0
epsilon = 0.0

batch_mean = x.mean()
batch_var = ((x - batch_mean) ** 2).mean()
y = gamma * (x - batch_mean) / np.sqrt(batch_var + epsilon) + beta

print(f"批量均值：{batch_mean:.4f}")
print(f"批量方差：{batch_var:.4f}")
print("批量归一化后的输出：")
for index, value in enumerate(y, start=1):
    print(f"y{index} = {value:.4f}")

批量均值：5.0000
批量方差：5.0000
批量归一化后的输出：
y1 = -1.6833
y2 = 0.1056
y3 = 1.8944
y4 = 3.6833


### 4.2 编程题：自定义残差块 Residual

残差块的核心是将卷积变换 $f(x)$ 与输入 $x$ 相加，再经过 ReLU：

$$
\operatorname{ReLU}(f(x)+x).
$$

当输入和输出的通道数或空间尺寸不一致时，使用 $1 \times 1$ 卷积调整输入，使两者能够按元素相加。

In [6]:
class Residual(nn.Module):
    """ResNet 中常用的基础残差块。"""

    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        # 第一层 3 x 3 卷积可以通过 strides 改变空间尺寸。
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        # 第二层 3 x 3 卷积保持空间尺寸不变。
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_channels)

        # 如果形状不一致，就用 1 x 1 卷积调整残差分支的通道数和空间尺寸。
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None

    def forward(self, x):
        """执行残差块前向传播。"""
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        y = y + x
        return F.relu(y)


# 情况 1：输入输出形状一致，不需要 1 x 1 卷积。
residual_same = Residual(input_channels=3, num_channels=3)
input_same = torch.randn(2, 3, 32, 32)
output_same = residual_same(input_same)

# 情况 2：输出通道数改变且空间尺寸减半，需要 1 x 1 卷积对齐形状。
residual_downsample = Residual(input_channels=3, num_channels=8, use_1x1conv=True, strides=2)
input_downsample = torch.randn(2, 3, 32, 32)
output_downsample = residual_downsample(input_downsample)

print("残差块测试结果：")
print(f"不使用 1x1 卷积时，输入形状：{tuple(input_same.shape)}，输出形状：{tuple(output_same.shape)}")
print(f"使用 1x1 卷积且 stride=2 时，输入形状：{tuple(input_downsample.shape)}，输出形状：{tuple(output_downsample.shape)}")
print("两个测试都说明 Residual 块可以完成 f(x) + x 的形状对齐和前向传播。")

残差块测试结果：
不使用 1x1 卷积时，输入形状：(2, 3, 32, 32)，输出形状：(2, 3, 32, 32)
使用 1x1 卷积且 stride=2 时，输入形状：(2, 3, 32, 32)，输出形状：(2, 8, 16, 16)
两个测试都说明 Residual 块可以完成 f(x) + x 的形状对齐和前向传播。


## 5 图像增广、微调和样式迁移

### 5.1 理论计算题

#### 1. 为什么底层特征提取层学习率较小，而顶层输出层学习率较大？

在 ImageNet 等大型源数据集上预训练得到的底层和中间层，通常已经学到了边缘、纹理、颜色、局部形状等比较通用的视觉特征。如果目标任务和源任务同属图像领域，这些特征往往仍然有价值。因此，对底层特征提取层使用较小学习率，甚至冻结参数，可以保留预训练知识，降低灾难性遗忘和过拟合风险。

相比之下，最终输出层通常需要根据目标数据集的类别数重新初始化。它一开始并没有学到目标任务的类别映射，所以需要较大的学习率，让它更快适应新的分类边界或回归目标。

#### 2. 目标数据集非常小且与源数据集非常相似时，如何微调以防止过拟合？

可以采取保守的微调策略：冻结大部分甚至全部特征提取层，只训练新初始化的输出层；如果验证集表现仍需提升，再只解冻靠近输出端的少数高层，并使用较小学习率微调。同时配合数据增广、权重衰减、Dropout、早停和验证集监控，避免模型在小数据集上记忆训练样本。

### 5.2 编程题：构建图像增广管道

下面使用 `torchvision.transforms` 创建组合图像增广管道。为了让示例不依赖外部图片，代码中会构造一张简单的 RGB 图像并通过增广管道。

In [7]:
# 按题目要求构建图像增广流水线。
train_transforms = transforms.Compose([
    # 随机裁剪图像，裁剪面积比例在 0.08 到 1.0 之间，并缩放到 224 x 224。
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0)),
    # 以 50% 的概率进行水平翻转。
    transforms.RandomHorizontalFlip(p=0.5),
    # 随机扰动亮度、对比度和饱和度，变化强度都设为 0.5。
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 转换为 PyTorch 张量，输出形状为 C x H x W，数值范围为 [0, 1]。
    transforms.ToTensor(),
])

# 构造一张不依赖外部文件的 RGB 示例图像。
height, width = 256, 256
x_axis = np.linspace(0, 255, width, dtype=np.uint8)
y_axis = np.linspace(0, 255, height, dtype=np.uint8)
red_channel = np.tile(x_axis, (height, 1))
green_channel = np.tile(y_axis[:, None], (1, width))
blue_channel = np.full((height, width), 128, dtype=np.uint8)
example_image = Image.fromarray(np.stack([red_channel, green_channel, blue_channel], axis=-1))

augmented_tensor = train_transforms(example_image)

print("图像增广管道已创建：随机裁剪缩放 -> 随机水平翻转 -> 颜色扰动 -> 转为张量。")
print(f"原始示例图像尺寸：{example_image.size}")
print(f"增广后张量形状：{tuple(augmented_tensor.shape)}")
print(f"增广后张量取值范围：最小值 {augmented_tensor.min().item():.4f}，最大值 {augmented_tensor.max().item():.4f}")

图像增广管道已创建：随机裁剪缩放 -> 随机水平翻转 -> 颜色扰动 -> 转为张量。
原始示例图像尺寸：(256, 256)
增广后张量形状：(3, 224, 224)
增广后张量取值范围：最小值 0.1569，最大值 0.4510


## 6 目标检测、计算机视觉训练技巧

### 6.1 理论计算题

真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$。

交集左上角坐标为

$$
(\max(10,30),\max(10,30))=(30,30).
$$

交集右下角坐标为

$$
(\min(50,70),\min(50,70))=(50,50).
$$

因此交集宽和高均为 20，交集面积为

$$
20\times20=400.
$$

两个框的面积分别为

$$
S_A=(50-10)(50-10)=1600,
$$

$$
S_B=(70-30)(70-30)=1600.
$$

并集面积为

$$
S_A+S_B-S_{intersection}=1600+1600-400=2800.
$$

所以 IoU 为

$$
\operatorname{IoU}=\frac{400}{2800}=\frac{1}{7}\approx 0.142857.
$$

In [8]:
def calculate_iou(box_a, box_b):
    """计算两个边界框的交并比，边界框格式为 [左上角x, 左上角y, 右下角x, 右下角y]。"""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # 计算交集矩形的坐标。
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    # 若两个框不相交，交集宽高应截断为 0。
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    inter_area = inter_width * inter_height

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area

    if union_area == 0:
        return 0.0
    return inter_area / union_area


box_a = [10, 10, 50, 50]
box_b = [30, 30, 70, 70]
iou_value = calculate_iou(box_a, box_b)

print(f"真实框 A：{box_a}")
print(f"预测框 B：{box_b}")
print(f"A 与 B 的 IoU：{iou_value:.6f}")
print(f"准确分数形式：1 / 7 = {1 / 7:.6f}")

真实框 A：[10, 10, 50, 50]
预测框 B：[30, 30, 70, 70]
A 与 B 的 IoU：0.142857
准确分数形式：1 / 7 = 0.142857


### 6.2 编程题：标签平滑交叉熵损失

标签平滑把真实类别的目标概率从 $1$ 调整为 $1-\epsilon$，把其余 $K-1$ 个错误类别的目标概率从 $0$ 调整为 $\frac{\epsilon}{K-1}$。这样可以避免模型对单一类别过度自信，提高泛化能力。

In [9]:
def label_smoothing_cross_entropy(logits, targets, epsilon=0.1, reduction="mean"):
    """
    计算标签平滑后的交叉熵损失。

    参数：
    logits：模型未归一化输出，形状为 (batch_size, num_classes)。
    targets：真实类别编号，形状为 (batch_size,)。
    epsilon：标签平滑因子。
    reduction：'mean' 返回平均损失，'sum' 返回总损失，'none' 返回逐样本损失。
    """
    if logits.ndim != 2:
        raise ValueError("logits 必须是二维张量，形状为 (batch_size, num_classes)。")
    if targets.ndim != 1:
        raise ValueError("targets 必须是一维张量，形状为 (batch_size,)。")
    if logits.shape[0] != targets.shape[0]:
        raise ValueError("logits 和 targets 的样本数量必须一致。")
    if not 0 <= epsilon < 1:
        raise ValueError("epsilon 必须满足 0 <= epsilon < 1。")

    num_classes = logits.shape[1]
    if num_classes < 2:
        raise ValueError("类别数必须至少为 2。")

    log_probs = F.log_softmax(logits, dim=1)

    # 构造平滑后的目标分布：真实类别为 1-epsilon，其余类别为 epsilon/(K-1)。
    with torch.no_grad():
        smooth_targets = torch.full_like(log_probs, fill_value=epsilon / (num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)

    losses = -(smooth_targets * log_probs).sum(dim=1)

    if reduction == "mean":
        return losses.mean()
    if reduction == "sum":
        return losses.sum()
    if reduction == "none":
        return losses
    raise ValueError("reduction 只能是 'mean'、'sum' 或 'none'。")


# 构造一个三分类示例，验证函数输出。
example_logits = torch.tensor([
    [2.0, 0.5, -1.0],
    [0.1, 1.2, 0.3],
], dtype=torch.float32)
example_targets = torch.tensor([0, 2], dtype=torch.long)

loss_each = label_smoothing_cross_entropy(example_logits, example_targets, epsilon=0.1, reduction="none")
loss_mean = label_smoothing_cross_entropy(example_logits, example_targets, epsilon=0.1, reduction="mean")

print("标签平滑交叉熵示例：")
print(f"logits 形状：{tuple(example_logits.shape)}")
print(f"真实标签：{example_targets.tolist()}")
print(f"逐样本损失：{loss_each.detach().numpy()}")
print(f"平均损失：{loss_mean.item():.6f}")
print("函数已经按照真实类别 1-epsilon、错误类别 epsilon/(K-1) 的规则构造平滑标签。")

标签平滑交叉熵示例：
logits 形状：(2, 3)
真实标签：[0, 2]
逐样本损失：[0.4663 1.4186]
平均损失：0.942437
函数已经按照真实类别 1-epsilon、错误类别 epsilon/(K-1) 的规则构造平滑标签。


## 小结

本作业已经完成全部题目：

- 给出卷积输出尺寸、卷积乘法次数、VGG 参数量、Batch Normalization、IoU 和微调策略的理论计算与解释；
- 使用 NumPy 手动实现支持步幅与填充的二维最大池化；
- 使用 PyTorch 定义 NiN 块和 Residual 残差块，并验证输入输出形状；
- 使用 `torchvision.transforms` 构建图像增广管道；
- 实现标签平滑交叉熵损失函数，并给出示例输出。